In [ ]:
import pandas as pd
import json
from pathlib import Path
import numpy as np
import librosa


with open('Data/splits_emotion_recognition.json', 'r') as f:
    data = json.load(f)

# Flatten nested structures
df = pd.DataFrame(data["train"])


df["author"] = df["song_author"].apply(lambda x: x[0] if x else None)



In [103]:
df.loc[len(df)] = [np.nan, "Devil In a New Dress", 0, "Joy", "personal", 0, 32897, ["Kanye West"], "Kanye West", 1]

In [11]:
df.emotion.unique()

array(['Amusement', 'Disappointment', 'Anger', 'Fear', 'Joy', 'Interest',
       'Contentment', 'Compassion', 'Contempt', 'Disgust', 'Love',
       'Pride', 'Relief', 'Sadness', 'Pleasure', 'Guilt', 'Regret',
       'Admiration', 'Hate'], dtype=object)

In [92]:
#1 is positive, 0 is negative
EMOTION_TO_INT = {
    "Amusement": 1,
    'Disappointment': 0, 
    'Anger': 0,
    'Fear': 0,
    'Joy': 1, 
    'Interest': 1,
    'Contentment': 1,
    'Compassion': 1,
    'Contempt': 0,
    'Disgust': 0,
    'Love': 1,
    'Pride': 1,
    'Relief': 1,
    'Sadness': 0,
    'Pleasure': 1,
    'Guilt': 0,
    'Regret': 0,
    'Admiration': 1,
    'Hate': 0
}

In [93]:
df["Y"] = df["emotion"].map(EMOTION_TO_INT)

In [104]:
ys = []

for song in set(df["song_title"]):
    if Path(f"Final_data/audio_data/{song}.mp3").exists():
        print(f"Found audio file for {song}")
        ys.append(df[df["song_title"] == song]["Y"].values[0])


Found audio file for Fallin'
Found audio file for More Than You Know
Found audio file for A Thousand Years
Found audio file for Memory
Found audio file for Everyday Life
Found audio file for What I've Done
Found audio file for Mica Van Gogh
Found audio file for Cold Cold Man
Found audio file for My Heart Is Broken
Found audio file for Burden In My Hand
Found audio file for Time
Found audio file for Ridere
Found audio file for On The Mend
Found audio file for My Way
Found audio file for Ho amato tutto
Found audio file for Stupido Hotel
Found audio file for The Show Must Go On - Remastered 2011
Found audio file for Let Her Go
Found audio file for Numb
Found audio file for Part Of Me
Found audio file for Devil In a New Dress
Found audio file for Bella
Found audio file for Il mondo è mio
Found audio file for In the End
Found audio file for Pyramid Song
Found audio file for Good Enough
Found audio file for Vikings Attack


In [96]:
len(ys)

26

In [ ]:

def extract_music_features(audio_path):
    y, sr = librosa.load(audio_path, duration=30)  # 30s clip
    
    features = {}
    features["name"] = Path(audio_path).stem
    
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
    features['tempo']            = tempo[0]
    features['beat_strength']    = np.mean(librosa.onset.onset_strength(y=y, sr=sr))
    
    rms = librosa.feature.rms(y=y)
    features['energy_mean']      = np.mean(rms)
    features['energy_std']       = np.std(rms)
    features['dynamic_range']    = np.max(rms) - np.min(rms)
    
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    features['chroma_mean']      = np.mean(chroma)
    features['chroma_std']       = np.std(chroma)
    
    features['spectral_centroid']  = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    features['spectral_contrast']  = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr))
    features['spectral_rolloff']   = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    features['spectral_flatness']  = np.mean(librosa.feature.spectral_flatness(y=y))
    
    # MFCCs — useful
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i}_mean'] = np.mean(mfcc[i])
        features[f'mfcc_{i}_std']  = np.std(mfcc[i])
    
    features['zcr_mean'] = np.mean(librosa.feature.zero_crossing_rate(y=y))
    
    return features

In [106]:
featahs = []
for song in df["song_title"]:
    path = f"Final_data/audio_data/{song}.mp3"
    if Path(path).exists():
        label = df[df["song_title"] == song]["Y"].values[0]
        featah = extract_music_features(path)
        featah["label"] = label
        featahs.append(featah)


In [107]:
songs = pd.DataFrame(featahs)

In [108]:
songs.to_csv("Final_data/songs_features.csv", index=False)

In [100]:
songs

,name,tempo,beat_strength,energy_mean,energy_std,dynamic_range,chroma_mean,chroma_std,spectral_centroid,spectral_contrast,...,mfcc_9_mean,mfcc_9_std,mfcc_10_mean,mfcc_10_std,mfcc_11_mean,mfcc_11_std,mfcc_12_mean,mfcc_12_std,zcr_mean,label
0,Everyday Life,123.046875,0.981157,0.025713,0.014985,0.083676,0.408541,0.258434,1537.731119,23.296503,...,-8.680209,6.275540,-15.094731,6.271287,-0.864311,7.043264,-0.788505,6.942981,0.098072,0
1,Burden In My Hand,123.046875,1.336275,0.041835,0.017012,0.068825,0.459583,0.265204,2551.869913,22.234506,...,-3.696764,8.544387,-3.825222,8.088765,-3.495393,8.881334,-9.999342,8.534403,0.147812,0
2,Pyramid Song,67.999589,0.953440,0.128004,0.069231,0.303919,0.288982,0.279187,839.823618,23.027056,...,-7.560100,6.494928,-10.212765,6.018682,-8.773573,5.335854,-8.033224,5.323475,0.036183,0
3,On The Mend,135.999178,1.201808,0.222142,0.058195,0.413075,0.316370,0.253531,1527.738359,24.206465,...,3.848006,7.888925,-13.473984,8.385788,1.724091,5.559725,-0.029374,6.815355,0.045591,1
4,Mica Van Gogh,103.359375,1.279580,0.159782,0.052167,0.287848,0.525016,0.246945,3453.818105,20.013816,...,3.799896,7.767109,-0.058388,7.153408,2.393339,7.260056,-3.228938,6.573883,0.191171,1
5,Fallin',95.703125,1.114936,0.080696,0.069389,0.349992,0.353420,0.308956,1746.355846,24.132084,...,3.597586,11.018093,-15.395278,15.652665,-2.110955,8.628671,-4.354525,10.935416,0.082484,1
6,Memory,161.499023,1.221232,0.027609,0.011699,0.070895,0.232566,0.283031,1834.786700,26.533982,...,5.230674,12.405570,-12.200033,9.163970,-2.326461,7.898326,-5.201286,8.771672,0.067713,1
7,Stupido Hotel,95.703125,1.359977,0.059458,0.047080,0.241977,0.351079,0.294441,2610.528507,25.038974,...,17.472822,12.026297,-3.653404,8.513494,8.813751,9.350851,0.115426,8.713865,0.096192,0
8,Bella,99.384014,1.560547,0.201710,0.075903,0.417583,0.547535,0.246632,2085.810576,22.968900,...,-0.317312,9.009004,-3.022665,9.670026,3.843270,8.885162,-1.574325,7.657412,0.078701,1
9,Il mondo è mio,57.421875,1.012842,0.027436,0.023914,0.102701,0.298446,0.287715,1278.815868,24.638132,...,-3.196683,8.276055,-6.889019,8.281313,-4.578111,9.037281,-5.687866,8.543516,0.053243,1
